In [1]:
import time
import requests
import pandas as pd
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional, Set

API_URL = "https://api.hyperliquid.xyz/info"

# -------------------------------------------------------------------------
# LISTA TICKER (stessa lista del funding)
# -------------------------------------------------------------------------
tickers = [
    'ATOM','REQ','CRV','MAVIA','SAGA','NEAR','MORPHO','MANTA','MOVE','XAI',
    'ETC','DOGE','SOPH','CELO','MAV','POPCAT','SCR','COMP','GMT','SOL','IMX',
    'JUP','RUNE','LAUNCHCOIN','UMA','TRB','USTC','AIXBT','IOTA','VIRTUAL',
    'ALGO','GMX','ANIME','BCH','BIO','BSV','NXPC','MOODENG','TNSR','HBAR',
    'SNX','ZEREBRO','HYPER','SAND','BERA','PURR','GAS','LDO','ONDO','DYDX',
    'FTT','TON','EIGEN','LTC','BLAST','AI16Z','OMNI','AAVE','OGN','SUI',
    'MEME','FXS','NEIROETH','NIL','CFX','ME','XRP','TIA','BNB','NOT','IP',
    'OM','TAO','OP','CAKE','AVAX','kPEPE','GALA','MNT','BOME','SUPER','SEI',
    'VINE','KAS','BABY','STX','S','FARTCOIN','STG','RENDER','ENA','LINK',
    'ARB','ARK','BIGTIME','BTC','ETH','RSR','kDOGS','BRETT','BANANA','XLM',
    'INJ','ENS','AR','DOT','SPX','ETHFI','PAXG','kLUNC','GOAT','kSHIB','FIL',
    'MEW','STRK','TRX','ZK','KAITO','PENGU','kBONK','VVV','ORDI','INIT','APT',
    'REZ','LAYER','ZEN','SUSHI','kFLOKI','ADA','kNEIRO','PEOPLE','ZORA',
    'PENDLE','APE','HYPE','FET','CHILLGUY','MELANIA','GRIFFAIN','PNUT','DOOD',
    'WIF','ACE','ZETA','TRUMP','NEO','JTO','YGG','ZRO','PROMPT','WLD','W',
    'MERL','BLUR','UNI','DYM','MINA','MKR','POLYX','POL','IO','TURBO','PYTH',
    'USUAL','GRASS','ALT','HMSTR','WCT','SYRUP','RESOLV','PROVE','YZY','WLFI',
    'TST','PUMP','LINEA','SKY','ASTER','0G','STBL','AVNT','XPL','ZEC','ICP'
]


# -------------------------------------------------------------------------
# CONFIG
# -------------------------------------------------------------------------

DATA_DIR = Path("../data")

# Output separato dal vecchio perps_prices_19.csv per non mischiare dati daily e hourly.
OUT_NAME = "perps_prices_1h_ohlc.csv"

CANDLE_INTERVAL = "1h"
CANDLE_MS = 60 * 60 * 1000

# La doc Hyperliquid indica che le risposte con time range vanno paginate.
# Uso chunk <= 500 candele per stare largo e non perdere righe su range lunghi.
MAX_CANDLES_PER_REQUEST = 500

RESUME_EPSILON_MS = 1  # evita di riscaricare l'ultima riga già salvata
SLEEP_BETWEEN_REQUESTS = 0.20


# -------------------------------------------------------------------------
# FUNZIONI DI SUPPORTO
# -------------------------------------------------------------------------

def safe_post(payload: dict,
              max_retries: int = 5,
              base_delay: float = 1.0) -> Optional[requests.Response]:
    """
    POST verso API_URL gestendo rate-limit (429) con retry e backoff esponenziale.
    """
    attempt = 0
    delay = base_delay

    while True:
        try:
            r = requests.post(API_URL, json=payload, timeout=10)

            if r.status_code == 429:
                attempt += 1
                if attempt > max_retries:
                    print(f"Rate limit esaurito per payload={payload.get('type')} {payload.get('coin')}, rinuncio.")
                    return None
                print(f"Rate limited (429) per {payload.get('type')} {payload.get('coin')}, retry fra {delay:.1f}s...")
                time.sleep(delay)
                delay *= 2
                continue

            r.raise_for_status()
            return r

        except Exception as e:
            attempt += 1
            if attempt > max_retries:
                print(f"HTTP error definitivo per {payload.get('type')} {payload.get('coin')}: {e}")
                return None
            print(f"HTTP error per {payload.get('type')} {payload.get('coin')}: {e}, retry fra {delay:.1f}s...")
            time.sleep(delay)
            delay *= 2


def get_perp_universe() -> Optional[Set[str]]:
    """
    Restituisce l'insieme dei nomi dei perps (campo 'name' in 'universe')
    usando type: 'meta'. Se fallisce, restituisce None.
    """
    payload = {"type": "meta"}
    r = safe_post(payload)
    if r is None:
        print("Impossibile ottenere la meta; uso la lista ticker così com'è.")
        return None

    try:
        meta = r.json()
    except Exception as e:
        print(f"Errore parsing meta JSON: {e} | raw: {r.text[:200]}")
        return None

    universe = meta.get("universe", [])
    names = {c.get("name") for c in universe if isinstance(c, dict) and "name" in c}
    return names


def detect_listing_time_ms(coin: str, end_ms: int) -> Optional[int]:
    """
    Stima la 'listing date' del perp come il timestamp del primo record
    disponibile in fundingHistory.
    """
    payload = {
        "type": "fundingHistory",
        "coin": coin,
        "startTime": 0,
        "endTime": end_ms,
    }

    r = safe_post(payload)
    if r is None:
        print(f"Impossibile detectare listing per {coin} (safe_post fallita).")
        return None

    try:
        data = r.json()
    except Exception as e:
        print(f"JSON decode error per listing {coin}: {e} | raw: {r.text[:200]}")
        return None

    if not isinstance(data, list) or len(data) == 0:
        return None

    first = data[0]
    t = first.get("time")
    if t is None:
        return None

    return int(t)


def fetch_candles_once(coin: str,
                       start_ms: int,
                       end_ms: int,
                       interval: str = CANDLE_INTERVAL) -> pd.DataFrame:
    """
    Scarica un singolo chunk di candele per `coin` tra start_ms ed end_ms.
    """
    payload = {
        "type": "candleSnapshot",
        "req": {
            "coin": coin,
            "interval": interval,
            "startTime": start_ms,
            "endTime": end_ms
        }
    }

    r = safe_post(payload)
    if r is None:
        return pd.DataFrame()

    try:
        data = r.json()
    except Exception as e:
        print(f"JSON decode error per candles {coin}: {e} | raw: {r.text[:200]}")
        return pd.DataFrame()

    if not isinstance(data, list) or len(data) == 0:
        return pd.DataFrame()

    df = pd.DataFrame(data)

    # Colonne tipiche: t, T, s, i, o, c, h, l, v, n
    if "t" not in df.columns:
        print(f"Colonne inattese per candles {coin}: {df.columns.tolist()}")
        return pd.DataFrame()

    df["time"] = pd.to_datetime(df["t"], unit="ms", utc=True)
    return df


def fetch_candles_paginated(coin: str,
                            start_ms: int,
                            end_ms: int,
                            interval: str = CANDLE_INTERVAL,
                            max_candles_per_request: int = MAX_CANDLES_PER_REQUEST) -> pd.DataFrame:
    """
    Scarica tutte le candele orarie paginando il range temporale.

    Il vecchio codice faceva una sola chiamata candleSnapshot: su range lunghi
    questo può troncare i dati. Qui il range viene diviso in blocchi da
    max_candles_per_request ore.
    """
    if start_ms >= end_ms:
        return pd.DataFrame()

    chunk_ms = max_candles_per_request * CANDLE_MS
    cursor_ms = start_ms
    chunks = []

    while cursor_ms < end_ms:
        chunk_end_ms = min(cursor_ms + chunk_ms, end_ms)

        df_chunk = fetch_candles_once(
            coin=coin,
            start_ms=cursor_ms,
            end_ms=chunk_end_ms,
            interval=interval,
        )

        if not df_chunk.empty:
            chunks.append(df_chunk)

        cursor_ms = chunk_end_ms
        time.sleep(SLEEP_BETWEEN_REQUESTS)

    if not chunks:
        return pd.DataFrame()

    df = pd.concat(chunks, ignore_index=True)
    df = (
        df
        .drop_duplicates(subset=["t"], keep="last")
        .sort_values("t")
        .reset_index(drop=True)
    )
    return df


# -------------------------------------------------------------------------
# RESUME / APPEND SU CSV ESISTENTE
# -------------------------------------------------------------------------

def load_existing_csv(out_path: Path) -> pd.DataFrame:
    """
    Carica il CSV già presente in data/, se esiste.
    La colonna time viene normalizzata in UTC per calcolare il resume.
    """
    if not out_path.exists():
        return pd.DataFrame()

    try:
        df = pd.read_csv(out_path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()

    if "time" in df.columns:
        df["time"] = pd.to_datetime(df["time"], utc=True, errors="coerce")
        df = df.dropna(subset=["time"])

    return df


def get_last_time_by_perp(existing_df: pd.DataFrame) -> dict:
    """
    Restituisce, per ogni perp, l'ultimo timestamp già salvato.
    Così ogni ticker riparte dal proprio ultimo dato, non da una data globale.
    """
    required_cols = {"perp", "time"}
    if existing_df.empty or not required_cols.issubset(existing_df.columns):
        return {}

    return existing_df.groupby("perp")["time"].max().to_dict()


def ts_to_ms(ts) -> int:
    """Converte un pandas/datetime timestamp in millisecondi Unix."""
    return int(pd.Timestamp(ts).timestamp() * 1000)


def append_dedup_and_save(
    new_df: pd.DataFrame,
    existing_df: pd.DataFrame,
    out_path: Path,
    dedup_cols=("perp", "time"),
) -> int:
    """
    Appende i nuovi dati ai vecchi, rimuove eventuali duplicati e salva in data/.
    """
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    if existing_df.empty:
        combined = new_df.copy()
    else:
        combined = pd.concat([existing_df, new_df], ignore_index=True)

    combined["time"] = pd.to_datetime(combined["time"], utc=True, errors="coerce")
    combined = combined.dropna(subset=["perp", "time"])
    combined = (
        combined
        .drop_duplicates(subset=list(dedup_cols), keep="last")
        .sort_values(["perp", "time"])
    )

    # Normalizza le colonne numeriche OHLCV.
    for col in ["open", "high", "low", "close", "volume", "num_trades"]:
        if col in combined.columns:
            combined[col] = pd.to_numeric(combined[col], errors="coerce")

    combined.to_csv(out_path, index=False)
    return len(combined)


# -------------------------------------------------------------------------
# MAIN: scarica prezzi orari OHLC, non solo le 19:00
# -------------------------------------------------------------------------

def main():
    # Start "globale" minimo (come nel funding)
    global_start = datetime(2025, 6, 5, tzinfo=timezone.utc)
    now = datetime.now(timezone.utc)

    global_start_ms = int(global_start.timestamp() * 1000)
    end_ms = int(now.timestamp() * 1000)

    out_path = DATA_DIR / OUT_NAME
    existing_df = load_existing_csv(out_path)
    last_time_by_perp = get_last_time_by_perp(existing_df)

    if existing_df.empty:
        print(f"CSV esistente non trovato o vuoto: {out_path}. Download completo dal global_start.")
    else:
        print(
            f"CSV esistente: {out_path} | righe={len(existing_df)} | "
            f"ultimo time={existing_df['time'].max()}"
        )

    # 1) Universo dei perps da HL
    perp_universe = get_perp_universe()
    if perp_universe is not None:
        valid_tickers = [t for t in tickers if t in perp_universe]
        missing = sorted(set(tickers) - perp_universe)
        if missing:
            print("Ticker non trovati in universe (probabilmente non perps o nomi diversi):")
            print(", ".join(missing))
    else:
        valid_tickers = tickers

    price_rows = []

    for ticker in valid_tickers:
        print(f"\n=== {ticker} ===")

        # 2) Detect listing time per questo perp (come nel funding)
        listing_ms = detect_listing_time_ms(ticker, end_ms)
        if listing_ms is None:
            print(f"⚠️ Nessun funding (o impossibile trovare listing) per {ticker} — skipped prezzi")
            time.sleep(SLEEP_BETWEEN_REQUESTS)
            continue

        # Start effettivo: max(start globale, listing, ultimo dato già salvato + 1 ms)
        start_ms = max(global_start_ms, listing_ms)
        last_saved_time = last_time_by_perp.get(ticker)

        if last_saved_time is not None and not pd.isna(last_saved_time):
            resume_ms = ts_to_ms(last_saved_time) + RESUME_EPSILON_MS
            start_ms = max(start_ms, resume_ms)

        if start_ms >= end_ms:
            print(f"Già aggiornato fino a {last_saved_time} — skipped")
            time.sleep(SLEEP_BETWEEN_REQUESTS)
            continue

        print(
            f"Listing {ticker}: {datetime.fromtimestamp(listing_ms/1000, tz=timezone.utc)} "
            f"| ultimo salvato: {last_saved_time} "
            f"| start effettivo_candles: {datetime.fromtimestamp(start_ms/1000, tz=timezone.utc)}"
        )

        # 3) Scarica TUTTE le candele 1h dall'effettivo start_ms a end_ms
        df_c = fetch_candles_paginated(ticker, start_ms, end_ms, interval=CANDLE_INTERVAL)

        if df_c.empty:
            print(f"⚠️ Nessuna candela in range per {ticker} — skipped")
            time.sleep(SLEEP_BETWEEN_REQUESTS)
            continue

        # 4) Salva ogni candela oraria, non solo le 19:00.
        # Per backtest liquidazioni è meglio avere OHLC:
        # - short: usa high della candela per verificare se è stato toccato il liquidation price
        # - long: usa low della candela per verificare se è satato toccato il liquidation price
        for _, row in df_c.iterrows():
            price_rows.append({
                "perp": ticker,
                "time": row["time"],
                "open": row.get("o"),
                "high": row.get("h"),
                "low": row.get("l"),
                "close": row.get("c"),
                "volume": row.get("v"),
                "num_trades": row.get("n"),
            })

        print(f"Scaricate {len(df_c)} candele 1h per {ticker}")
        time.sleep(SLEEP_BETWEEN_REQUESTS)

    if not price_rows:
        print("Nessun nuovo dato prezzi da scaricare.")
        return

    final_df = pd.DataFrame(price_rows)
    final_df = final_df.sort_values(["perp", "time"])

    total_rows = append_dedup_and_save(final_df, existing_df, out_path)
    print(
        f"\n✔️ Appended + deduplicated → {out_path} "
        f"| righe totali={total_rows} | nuove righe={len(final_df)}"
    )


if __name__ == "__main__":
    main()


CSV esistente: ../data/perps_prices_1h_ohlc.csv | righe=907451 | ultimo time=2026-04-30 16:00:00+00:00

=== ATOM ===
Listing ATOM: 2023-05-12 00:00:00.048000+00:00 | ultimo salvato: 2026-04-30 16:00:00+00:00 | start effettivo_candles: 2026-04-30 16:00:00.001000+00:00
Scaricate 217 candele 1h per ATOM

=== REQ ===
Listing REQ: 2023-10-11 14:00:00.116000+00:00 | ultimo salvato: 2025-08-04 09:00:00+00:00 | start effettivo_candles: 2025-08-04 09:00:00.001000+00:00
Scaricate 1 candele 1h per REQ

=== CRV ===
Listing CRV: 2023-05-15 16:00:00.409000+00:00 | ultimo salvato: 2026-04-30 16:00:00+00:00 | start effettivo_candles: 2026-04-30 16:00:00.001000+00:00
Scaricate 217 candele 1h per CRV

=== MAVIA ===
Listing MAVIA: 2024-02-06 23:00:00.019000+00:00 | ultimo salvato: 2026-04-30 16:00:00+00:00 | start effettivo_candles: 2026-04-30 16:00:00.001000+00:00
Scaricate 114 candele 1h per MAVIA

=== SAGA ===
Listing SAGA: 2024-04-21 07:00:00.074000+00:00 | ultimo salvato: 2026-04-30 16:00:00+00:00 |